In [29]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

## 1. Load and combine

Train and test are combined so the group-survival feature can see every member of a family/ticket group. Test `Survived` is NaN and is never used as a label.

In [30]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

train["is_train"] = 1
test["is_train"] = 0
test["Survived"] = np.nan

data = pd.concat([train, test], ignore_index=True, sort=False)
data.shape

(1309, 13)

In [31]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Survived     891 non-null    float64
 2   Pclass       1309 non-null   int64  
 3   Name         1309 non-null   str    
 4   Sex          1309 non-null   str    
 5   Age          1046 non-null   float64
 6   SibSp        1309 non-null   int64  
 7   Parch        1309 non-null   int64  
 8   Ticket       1309 non-null   str    
 9   Fare         1308 non-null   float64
 10  Cabin        295 non-null    str    
 11  Embarked     1307 non-null   str    
 12  is_train     1309 non-null   int64  
dtypes: float64(3), int64(5), str(5)
memory usage: 133.1 KB


In [32]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,is_train
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,1
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,1
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1


## 2. Title and Surname

In [33]:
def extract_title(name):
    title = name.split(",")[1].split(".")[0].strip()
    title = {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}.get(title, title)
    if title not in {"Mr", "Mrs", "Miss", "Master"}:
        title = "Rare"
    return title

data["Title"] = data["Name"].apply(extract_title)
data["Surname"] = data["Name"].apply(lambda n: n.split(",")[0].strip())

## 3. Missing values

- **Age** imputed by the *median age of each Title* (so children with a missing age aren't given an adult age).
- **Fare** imputed by the *Pclass median* (the one missing fare is 3rd class).
- **Embarked** filled with the most common port, S.

In [34]:
data["Age"] = data.groupby("Title")["Age"].transform(lambda s: s.fillna(s.median()))
data["Fare"] = data.groupby("Pclass")["Fare"].transform(lambda s: s.fillna(s.median()))
data["Embarked"] = data["Embarked"].fillna("S")

## 4. Family / group survival rate

For each passenger, look at the **known** survival of the *other* members of their group (grouped by Surname+Fare, then by Ticket). The passenger's own label is dropped, so there is no leakage on the training rows. Unknown groups default to 0.5.

In [35]:
data["Family_Survival"] = 0.5

# Pass 1: group by Surname + Fare (families share both)
for _, grp in data.groupby(["Surname", "Fare"]):
    if len(grp) > 1:
        for ind in grp.index:
            others = grp.drop(ind)["Survived"]
            if others.max() == 1.0:
                data.loc[ind, "Family_Survival"] = 1
            elif others.min() == 0.0:
                data.loc[ind, "Family_Survival"] = 0

# Pass 2: refine with shared Ticket groups (catches relatives with different surnames)
for _, grp in data.groupby("Ticket"):
    if len(grp) > 1:
        for ind in grp.index:
            if data.loc[ind, "Family_Survival"] != 1:
                others = grp.drop(ind)["Survived"]
                if others.max() == 1.0:
                    data.loc[ind, "Family_Survival"] = 1
                elif others.min() == 0.0:
                    data.loc[ind, "Family_Survival"] = 0

data["Family_Survival"].value_counts()

Family_Survival
0.5    813
1.0    294
0.0    202
Name: count, dtype: int64

## 5. Engineered features

Family size + bins, per-person fare (log), age bins, and interaction terms a linear model cannot otherwise express.

In [36]:
# Family size and binning
data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
data["IsAlone"] = (data["FamilySize"] == 1).astype(int)
data["FamilyBin"] = pd.cut(data["FamilySize"], bins=[0, 1, 4, 20],
                           labels=["Alone", "Small", "Large"])

# Per-person fare, log-scaled
data["Fare_Per_Person"] = np.log1p(data["Fare"] / data["FamilySize"])

# Child flag + binning
data["IsChild"] = (data["Age"] < 16).astype(int)
data["AgeBin"] = pd.cut(data["Age"], bins=[0, 12, 18, 35, 60, 200],
                        labels=["Child", "Teen", "Adult", "MidAge", "Senior"])

data["is_male"] = (data["Sex"] == "male").astype(int)
data["Sex_Pclass"] = data["is_male"] * data["Pclass"]
data["Age_Pclass"] = data["Age"] * data["Pclass"]
data["FarePP_Pclass"] = data["Fare_Per_Person"] * data["Pclass"]

data["Title_Pclass"] = data["Title"] + "_" + data["Pclass"].astype(str)

# Cabin known/unknown flag
data["HasCabin"] = data["Cabin"].notna().astype(int)

## 6. Encode and split back

Drop redundant/collinear raw columns, one-hot the categoricals, then split the combined frame back into train and test.

In [37]:
data.drop(["Name", "Surname", "Ticket", "Cabin", "Sex", "SibSp", "Parch", "Fare"],
          axis=1, inplace=True)

data = pd.get_dummies(data, columns=["Embarked", "Title", "Pclass",
                                     "FamilyBin", "AgeBin", "Title_Pclass"])

train_proc = data[data["is_train"] == 1].copy()
test_proc = data[data["is_train"] == 0].copy()

X = train_proc.drop(["PassengerId", "Survived", "is_train"], axis=1)
y = train_proc["Survived"].astype(int)

X_test = test_proc.drop(["PassengerId", "Survived", "is_train"], axis=1)
X_test = X_test.reindex(columns=X.columns, fill_value=0)
X.shape, X_test.shape

((891, 44), (418, 44))

## 7. Fit and predict

`LinearRegression` as a linear probability model, fixed 0.5 cutoff.

In [38]:
linear_model = LinearRegression()
linear_model.fit(X, y)

best_threshold = 0.5
train_acc = ((linear_model.predict(X) > best_threshold).astype(int) == y).mean()
print(f"Train accuracy at threshold {best_threshold}: {train_acc:.3f}")

Train accuracy at threshold 0.5: 0.851


In [39]:
test_pred = (linear_model.predict(X_test) > best_threshold).astype(int)

submission = pd.DataFrame({
    "PassengerId": test_proc["PassengerId"].astype(int).values,
    "Survived": test_pred,
})
submission.to_csv("output2.csv", index=False)
submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
